# Archiving a Snapshot for Publication

Part of the [PyMAUDE](../) examples — see [`quickstart.ipynb`](../quickstart.ipynb) first if you haven't loaded a database yet.

MAUDE is updated continuously and isn't versioned, so a query run today may return different results in a year. `db.archive()` freezes the exact database backing an analysis so it can be cited or uploaded alongside a paper (e.g. to Zenodo):

- Checkpoints and copies the DuckDB file itself.
- Writes a `manifest.json` recording, per loaded table/year: source file, SHA-256 checksum, row count, and load timestamp — plus the DuckDB and pymaude versions used to build it.
- With `include_raw=True`, also copies the raw MAUDE source files referenced in the manifest into `output_dir/raw/`, so reviewers can re-derive the database from scratch.

---
## Contents
1. [Setup](#1-setup)
2. [Archive a snapshot](#2-archive)
3. [Inspect the manifest](#3-manifest)

---
## 1. Setup <a id="1-setup"></a>

In [ ]:
from pymaude import MaudeDatabase

DB_PATH  = '../maude.duckdb'
DATA_DIR = '../maude_data'
YEARS    = '2024-2026'

db = MaudeDatabase(DB_PATH, data_dir=DATA_DIR, verbose=True, memory_limit='2GB')
db.add_years(
    YEARS,
    tables=['master', 'device', 'text', 'patient', 'device_problem', 'patient_problem'],
    download=False
)

---
## 2. Archive a snapshot <a id="2-archive"></a>

In [ ]:
manifest_path = db.archive('./maude_archive', include_raw=False)
print(f'Archive written, manifest at: {manifest_path}')

---
## 3. Inspect the manifest <a id="3-manifest"></a>

In [ ]:
# Inspect the generated manifest — this is what accompanies the archived .duckdb file
import json

with open(manifest_path) as f:
    manifest = json.load(f)

print(f"pymaude {manifest['pymaude_version']}  ·  duckdb {manifest['duckdb_version']}  ·  {manifest['checksum_algorithm']}")
print(f"Database: {manifest['database']['filename']} ({manifest['database']['size_bytes']:,} bytes)")
print(f"Tables tracked: {len(manifest['tables'])}")
manifest['tables'][:5]

In [ ]:
db.close()